# Tutorial 4: Gene Regulatory Network Construction and Visualization
This notebook constructs a genome-scale gene regulatory network (GRN) for
regenerating planarians by predicting pairwise regulatory relationships
from learned gene embeddings. The approach couples two complementary data
modalities — PFM-T-derived gene expression embeddings capturing transcriptional
state, and perturbation response signatures encoding functional disruption
phenotypes — and feeds their concatenation into a multi-layer perceptron
trained to classify regulatory edges into seven ordered categories spanning
strong inhibition to strong activation.

The inference pipeline proceeds in three stages: **(1)** construction of a
labelled training set from known regulatory edges, **(2)** exhaustive pairwise
scoring of all gene combinations using a pre-trained MLP classifier, and
**(3)** filtering and export of high-confidence regulatory predictions for
downstream network visualisation and analysis.

In [1]:
import sys, os, itertools
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from xlsxwriter import Workbook
CODE_PATH = "/home/wwd/codebox/ODT-main/"
if CODE_PATH not in sys.path:
    sys.path.append(CODE_PATH)
from src.funcs import edge_class
from src.models import Graph_predicate

Training Data Preparation

`edge_target()` reads a curated set of known regulatory edges from a CSV file
and assigns each edge to one of seven ordered regulatory classes via
`edge_class()`, which bins the continuous edge weight into the following
schema: strong inhibition (0), moderate inhibition (1), weak inhibition (2),
neutral (3), weak activation (4), moderate activation (5), and strong
activation (6). This discretisation transforms a regression problem into a
seven-way ordinal classification task, which is substantially more tractable
for the downstream MLP.

`makexlsx()` is a utility that exports the top 200 source and target
regulators per gene into a formatted spreadsheet for manual inspection.

In [2]:
def edge_target():
    """Load known regulatory edges from CSV and discretise edge weights into
    seven ordinal regulatory classes via ``edge_class()``.
    """
    edge_pure = []
    edges_csv = os.path.join(CODE_PATH, 'results', 'GRN_edges', 'edges.csv')
    with open(edges_csv) as f:
        edges = f.readlines()
    for edge in edges[1:]:
        parts = edge.strip('\n').split(',')
        edge_pure.append([parts[1], parts[2], edge_class(float(parts[3]))])
    return edge_pure


def makexlsx(target_path, data_dict):
    """Export the top 200 source and target regulators per gene to an Excel
    workbook for manual inspection.
    """
    wb = Workbook()
    ws = wb.active
    ws.title = 'gene_all'
    ws.append([
        'Regenerative_genes', 'source_positive_gene', 'score',
        'source_negative_gene', 'score', 'target_positive_gene',
        'score', 'target_negative_gene', 'score'
    ])
    for R_gene in data_dict:
        for i in range(200):
            ws.append([
                R_gene,
                data_dict[R_gene]['source_positive'][i][0],
                data_dict[R_gene]['source_positive'][i][1],
                data_dict[R_gene]['source_negative'][i][0],
                data_dict[R_gene]['source_negative'][i][1],
                data_dict[R_gene]['target_positive'][i][0],
                data_dict[R_gene]['target_positive'][i][1],
                data_dict[R_gene]['target_negative'][i][0],
                data_dict[R_gene]['target_negative'][i][1]
            ])
    wb.save(target_path)

Exhaustive Pairwise Regulatory Prediction

`getRegulateAll()` is the core inference engine. It first instantiates a
`Graph_predicate` object, which loads two data structures for every gene in
the genome: a PFM-T-derived expression embedding and a perturbation response
vector summarising the transcriptional consequences of knocking down that
gene. For each ordered gene pair (A, B), the concatenated embedding
`[embedding_A || embedding_B]` is fed into a pre-trained three-hidden-layer
MLP that outputs a probability distribution over the seven regulatory classes.

Because the number of ordered gene pairs scales quadratically with the
genome size — approximately 8.6 million pairs for the ~4,150 genes in the
planarian embedding set — the function employs batched GPU inference with
incremental disk serialisation. Every 100,000 predictions, results are flushed
to a CSV partition, preventing memory exhaustion and enabling resumption from
the last completed batch if the process is interrupted.

Only edges whose predicted class deviates from the neutral midpoint (class 3)
by at least two categories are retained, corresponding to a stringent filter
for putative regulatory relationships with clear directional signal.

In [3]:
def getRegulateAll():
    """Perform exhaustive pairwise regulatory prediction across all genes
    using a pre-trained MLP classifier, with incremental disk serialisation
    every 100,000 predictions.
    """
    edge_predicate = edge_target()
    device = torch.device('cuda:9')

    # Instantiate the gene embedding and perturbation data container.
    gene_tensor_path = os.path.join(CODE_PATH, 'results', 'Embeddings', 'gene_embeddings.pkl')
    gene_disturb_path = os.path.join(CODE_PATH, 'results', 'Embeddings', 'planaria_disturb.pkl')
    Graph_predicate_example = Graph_predicate(
        gene_tensor_path=gene_tensor_path,
        gene_disturb_path=gene_disturb_path,
        target_edges=edge_predicate,
        device=device
    )

    # Load raw gene tensors (concatenation of embedding and perturbation vector).
    tensor_model = Graph_predicate_example.get_Tensor()
    model_dict = {
        'input_size': 548, 'num_classes': 7,
        'hidden_sizes': [640, 1280, 640],
        'num_epochs': 1000, 'dropout': 0.3
    }
    tensors = list(tensor_model.values())
    model_dict['input_size'] = tensors[0].shape[0] * 2
    print(f'Model input size: {model_dict["input_size"]}')

    # Load the pre-trained MLP checkpoint.
    mlp_model_path = os.path.join(CODE_PATH,'results', 'models', 'MLP_model2', 'mlp_model_checkpoint_epoch_1000.pth')
    Graph_predicate_example.loadMLPmodel(mlp_model_path, model_dict)
    Graph_predicate_example.mlpModel.eval()

    all_genes = list(Graph_predicate_example.gene_model_tensors.keys())
    gene_pairs = list(itertools.combinations(all_genes, 2))
    total_pairs = len(gene_pairs)
    print(f'Total gene pairs to score: {total_pairs:,}')

    batch_size = 12
    save_interval = 100_000
    batch_results = []
    save_count = 0

    with torch.no_grad():
        for i in tqdm(range(0, total_pairs, batch_size), total=total_pairs // batch_size):
            batch_pairs = gene_pairs[i:i + batch_size]
            source_tensors = []

            for gene1, gene2 in batch_pairs:
                tensor = torch.cat((
                    Graph_predicate_example.gene_model_tensors[gene1],
                    Graph_predicate_example.gene_model_tensors[gene2]
                ), dim=0).unsqueeze(0)
                source_tensors.append(tensor)

            source_tensors = torch.cat(source_tensors, dim=0).to(device)

            # Z-score standardise per batch.
            mean = source_tensors.mean(dim=1, keepdim=True)
            std = source_tensors.std(dim=1, keepdim=True)
            inputs_standardized = (source_tensors - mean) / (std + 1e-8)

            target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
            outs = torch.argmax(target_tensors, dim=1).tolist()

            # Retain only edges with clear regulatory signal (|class - 3| >= 2).
            for (gene1, gene2), out in zip(batch_pairs, outs):
                if abs(out - 3) >= 2:
                    batch_results.append([gene1, gene2, out])

            if len(batch_results) >= save_interval:
                df = pd.DataFrame(batch_results, columns=['Gene1', 'Gene2', 'Score'])
                out_path = os.path.join(CODE_PATH, 'results', 'GRN_edges', 'link_result', f'gene_link_part_{save_count}.csv')
                df.to_csv(out_path, index=False)
                save_count += 1
                batch_results = []
                
                # print(f'Partition {save_count} saved')

    # Flush remaining predictions.
    if batch_results:
        df = pd.DataFrame(batch_results, columns=['Gene1', 'Gene2', 'Score'])
        out_path = os.path.join(CODE_PATH, 'results', 'GRN_edges', 'link_result', f'gene_link_part_{save_count}.csv')
        df.to_csv(out_path, index=False)
        print(f'Final partition {save_count} saved')

    print('Exhaustive pairwise prediction complete.')

## Step 1: Perform exhaustive pairwise regulatory prediction across all genes

In [4]:
getRegulateAll()

/home/wwd/codebox/ODT-main/src/models.py:660: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.gene_model_tensors={gene_name:torch.tensor(self.gene_dict[gene_name]) for gene_name in self.gene_dict}


Model input size: 548
MLPmodel load success
Total gene pairs to score: 103,054,546


8587879it [2:57:52, 804.67it/s]                                


Final partition 274 saved
Exhaustive pairwise prediction complete.


## Step 2: Consolidate All Result Files

The CSV partitions produced by `getRegulateAll()` can be concatenated and
loaded into network analysis tools such as Cytoscape or NetworkX for
visualisation. Each row encodes a directed regulatory edge (source gene,
target gene, regulatory class), where the regulatory class maps to an
ordinal scale from strong inhibition (0) to strong activation (6). Filtering
by edge weight stringency and intersecting with known regeneration-associated
gene sets yields a high-confidence sub-network amenable to biological
interpretation and experimental validation.

In [5]:
# Concatenate all gene link partitions into a single edge table for visualisation.
import glob

link_dir = os.path.join(CODE_PATH, 'results', 'GRN_edges', 'link_result')
all_files = sorted(glob.glob(os.path.join(link_dir, 'gene_link_part_*.csv')))

dfs = []
for f in all_files:
    dfs.append(pd.read_csv(f))
full_network = pd.concat(dfs, ignore_index=True)
full_network.to_csv(os.path.join(link_dir, 'gene_link_all.csv'), index=False)

print(f'Consolidated {len(all_files)} partitions into gene_link_all.csv')
print(f'Total regulatory edges: {len(full_network):,}')
adjusted_labels = full_network["Score"] - 3
print(f'Edge class distribution:\n{adjusted_labels.value_counts().sort_index()}')

Consolidated 275 partitions into gene_link_all.csv
Total regulatory edges: 27,409,749
Edge class distribution:
Score
-3     2530270
-2     4159295
 2    15528231
 3     5191953
Name: count, dtype: int64
